In [0]:
from pyspark.sql.types import *

customer_schema = StructType([
    StructField("customer_id", StringType()),
    StructField("name", StringType()),
    StructField("address", StructType([
        StructField("city", StringType()),
        StructField("state", StringType())
    ])),
    StructField("orders", ArrayType(
        StructType([
            StructField("order_id", StringType()),
            StructField("amount", IntegerType())
        ])
    ))
])

In [0]:
df = (
    spark.read
    .schema(customer_schema)
    .json("/path/to/customers.json")
)

In [0]:
from pyspark.sql.functions import explode, col

flat_df = (
    df
    .withColumn("order", explode("orders"))
    .select(
        "customer_id",
        "name",
        col("address.city").alias("city"),
        col("address.state").alias("state"),
        col("order.order_id").alias("order_id"),
        col("order.amount").alias("amount")
    )
)

In [0]:
valid_df = flat_df.filter(
    "customer_id IS NOT NULL AND order_id IS NOT NULL"
)

In [0]:
valid_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "git_learning_lab.customer_pipeline.json_validations"
    )

In [0]:
valid_df = flat_df.filter(
    "customer_id IS NOT NULL AND order_id IS NOT NULL AND amount >= 100"
)